# BTP: Optical Transient Classification — Full Pipeline (TNS + TESS)

**Classes (8):** SN Ia, SN Ib, SN Ic, SN II, SLSN, AGN, TDE (all from TNS,
spectroscopically-confirmed, photometry via ZTF/ALeRCE), and Stellar Flares (TESS,
via lightkurve).

**Target sample sizes:** 30 objects per supernova subtype (Ia/Ib/Ic/II/SLSN — these
are intentionally smaller classes since fine-grained SN subtyping is a harder,
rarer-data problem), 150 objects each for AGN, TDE, and Stellar Flares.



# PHASE  2 — Data Acquisition (TNS + ZTF/ALeRCE + TESS)

In [ ]:
# Setup — install, mount, config
!pip install -q alerce lightkurve astroquery pandas numpy matplotlib tqdm requests

import os, io, json, time, zipfile, requests, gc
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from alerce.core import Alerce

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/BTP_optical_transients/data/raw'
FEATURE_DIR = '/content/drive/MyDrive/BTP_optical_transients/data/features'
for sub in ['sn_ia', 'sn_ib', 'sn_ic', 'sn_ii', 'slsn', 'agn', 'tde', 'stellar_flares', 'tns_catalog']:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)
os.makedirs(FEATURE_DIR, exist_ok=True)

alerce = Alerce()

# AGN / TDE / Stellar Flares target this many objects each
N_PER_CLASS = 150
# Each SN subtype (Ia/Ib/Ic/II/SLSN) targets this many objects — kept smaller
# since fine-grained subtyping has far less available data per subtype on TNS
N_PER_SN_SUBTYPE = 30

print('Data root:', BASE_DIR)


In [ ]:
# TNS credentials

TNS_MARKER = 'tns_marker{"tns_id":<REDACTED>,"type": "bot", "name":"<REDACTED>"}'
TNS_API_KEY = "<REDACTED-ROTATE-THIS-KEY>"

HEADERS = {'user-agent': TNS_MARKER}


In [ ]:
# Download the staged TNS public objects CSV (bulk file, exactly as TNS's approval
# email and docs ask for — NOT per-object queries, to keep load on their servers low)
TNS_CSV_URL = "https://www.wis-tns.org/system/files/tns_public_objects/tns_public_objects.csv.zip"

resp = requests.post(TNS_CSV_URL, headers=HEADERS, data={'api_key': TNS_API_KEY}, timeout=120)
resp.raise_for_status()

zip_path = os.path.join(BASE_DIR, 'tns_catalog', 'tns_public_objects.csv.zip')
with open(zip_path, 'wb') as f:
    f.write(resp.content)
with zipfile.ZipFile(zip_path) as z:
    csv_name = z.namelist()[0]
    z.extractall(os.path.join(BASE_DIR, 'tns_catalog'))

# First line of the file is a timestamp, not the header — confirmed against TNS's own docs
tns_full = pd.read_csv(os.path.join(BASE_DIR, 'tns_catalog', csv_name), skiprows=1)
print(tns_full.shape)
print(tns_full.columns.tolist())
tns_full.head()


In [ ]:
# Filter to SN Ia/Ib/Ic/II/SLSN, AGN and TDE, check class balance BEFORE downloading
print(tns_full['type'].value_counts().head(30))

def classify_sn_subtype(t):
    """
    Maps a raw TNS `type` string to one of five SN subtype buckets, or None if it
    doesn't belong to any of them. SLSN-I/II/R are TNS's own top-level type strings
    (not prefixed 'SN '), so they're checked separately from the SN Ia/Ib/Ic/II family.
    Sub-subtypes are folded into their parent bucket for a manageable class count:
    e.g. 'SN Ia-91bg', 'SN Ia-pec', 'SN Iax' -> SN_Ia; 'SN Ic-BL', 'SN Icn' -> SN_Ic;
    'SN IIb', 'SN IIn', 'SN IIP', 'SN IIL' -> SN_II.
    """
    t = str(t)
    if 'SLSN' in t:
        return 'SLSN'
    if not t.startswith('SN '):
        return None
    rest = t[3:]
    if rest.startswith('Ia'):
        return 'SN_Ia'
    if rest.startswith('Ib'):
        return 'SN_Ib'
    if rest.startswith('Ic'):
        return 'SN_Ic'
    if rest.startswith('II'):
        return 'SN_II'
    return None

tns_full['sn_subtype'] = tns_full['type'].apply(classify_sn_subtype)
agn_mask = tns_full['type'].astype(str).str.contains('AGN', case=False, na=False)
tde_mask = tns_full['type'].astype(str).str.contains('TDE', case=False, na=False)

SN_SUBTYPES = ['SN_Ia', 'SN_Ib', 'SN_Ic', 'SN_II', 'SLSN']
KEEP_COLS = ['name', 'ra', 'declination', 'redshift', 'type', 'discoverydate', 'internal_names']

tns_subtype_dfs = {}
for subtype in SN_SUBTYPES:
    df = tns_full[tns_full['sn_subtype'] == subtype].copy()
    df = df[[c for c in KEEP_COLS if c in df.columns]]
    print(f"TNS {subtype} rows available: {len(df)}")
    if len(df) < N_PER_SN_SUBTYPE:
        print(f"  NOTE: only {len(df)} available, below the target of {N_PER_SN_SUBTYPE} — "
              f"this class will end up smaller than the others.")
    df = df.sample(min(N_PER_SN_SUBTYPE, len(df)), random_state=42)
    df.to_csv(os.path.join(BASE_DIR, 'tns_catalog', f'tns_{subtype.lower()}_labels.csv'), index=False)
    tns_subtype_dfs[subtype] = df

tns_sn_ia = tns_subtype_dfs['SN_Ia']
tns_sn_ib = tns_subtype_dfs['SN_Ib']
tns_sn_ic = tns_subtype_dfs['SN_Ic']
tns_sn_ii = tns_subtype_dfs['SN_II']
tns_slsn  = tns_subtype_dfs['SLSN']

tns_agn = tns_full[agn_mask][[c for c in KEEP_COLS if c in tns_full.columns]].copy()
tns_tde = tns_full[tde_mask][[c for c in KEEP_COLS if c in tns_full.columns]].copy()
print(f"TNS AGN rows: {len(tns_agn)}")
print(f"TNS TDE rows: {len(tns_tde)}")
if len(tns_agn) < N_PER_CLASS:
    print(f"NOTE: only {len(tns_agn)} AGN available, below the target of {N_PER_CLASS}.")
if len(tns_tde) < N_PER_CLASS:
    print(f"NOTE: only {len(tns_tde)} TDE available, below the target of {N_PER_CLASS}.")

tns_agn = tns_agn.sample(min(N_PER_CLASS, len(tns_agn)), random_state=42)
tns_tde = tns_tde.sample(min(N_PER_CLASS, len(tns_tde)), random_state=42)
tns_agn.to_csv(os.path.join(BASE_DIR, 'tns_catalog', 'tns_agn_labels.csv'), index=False)
tns_tde.to_csv(os.path.join(BASE_DIR, 'tns_catalog', 'tns_tde_labels.csv'), index=False)

tns_sn_ia.head()


In [ ]:
def extract_ztf_name(internal_names):
    if not isinstance(internal_names, str):
        return None
    for tok in internal_names.split(','):
        tok = tok.strip()
        if tok.startswith('ZTF'):
            return tok
    return None

def resolve_oid(row, radius_arcsec=2.0):
    ztf_name = extract_ztf_name(row.get('internal_names'))
    if ztf_name:
        return ztf_name
    try:
        res = alerce.query_objects(ra=row['ra'], dec=row['declination'],
                                    radius=radius_arcsec, format='pandas')
        if len(res):
            return res.iloc[0]['oid']
    except Exception:
        pass
    return None

N_PER_SUBTYPE_TARGET = 30
OVERSAMPLE_FACTOR = 3
N_PER_SUBTYPE_SAMPLE = N_PER_SUBTYPE_TARGET * OVERSAMPLE_FACTOR

In [ ]:
# Resumable light curve download — checkpoints every 25 objects.
# If the kernel dies mid-loop, re-running this cell resumes automatically.
def fetch_lightcurves_from_tns(tns_df, label, out_subdir, sleep=0.15, checkpoint_every=25):
    out_dir = os.path.join(BASE_DIR, out_subdir)
    manifest_path = os.path.join(BASE_DIR, f'{out_subdir}_manifest_partial.csv')

    if os.path.exists(manifest_path):
        manifest = pd.read_csv(manifest_path).to_dict('records')
        done_oids = {m['oid'] for m in manifest}
        print(f"Resuming {label}: {len(done_oids)} already done")
    else:
        manifest = []
        done_oids = set()

    for count, (_, row) in enumerate(tqdm(tns_df.iterrows(), total=len(tns_df), desc=label)):
        oid = resolve_oid(row)
        if oid is None or oid in done_oids:
            continue
        try:
            det = alerce.query_detections(oid, format='pandas', sort='mjd')
        except Exception:
            continue
        if det is None or len(det) < 5:
            continue
        det = det[['mjd', 'fid', 'magpsf', 'sigmapsf', 'ra', 'dec']]
        det.to_csv(os.path.join(out_dir, f"{oid}.csv"), index=False)
        manifest.append({'tns_name': row['name'], 'oid': oid, 'label': label,
                          'n_points': len(det), 'redshift': row.get('redshift')})
        del det
        time.sleep(sleep)
        if count % checkpoint_every == 0:
            pd.DataFrame(manifest).to_csv(manifest_path, index=False)
            gc.collect()

    result = pd.DataFrame(manifest)
    result.to_csv(manifest_path, index=False)
    return result

sn_ia_manifest = fetch_lightcurves_from_tns(tns_sn_ia, 'SN_Ia', 'sn_ia')
sn_ib_manifest = fetch_lightcurves_from_tns(tns_sn_ib, 'SN_Ib', 'sn_ib')
sn_ic_manifest = fetch_lightcurves_from_tns(tns_sn_ic, 'SN_Ic', 'sn_ic')
sn_ii_manifest = fetch_lightcurves_from_tns(tns_sn_ii, 'SN_II', 'sn_ii')
slsn_manifest  = fetch_lightcurves_from_tns(tns_slsn,  'SLSN',  'slsn')
agn_manifest   = fetch_lightcurves_from_tns(tns_agn,   'AGN',   'agn')
tde_manifest   = fetch_lightcurves_from_tns(tns_tde,   'TDE',   'tde')

sn_ia_manifest.to_csv(os.path.join(BASE_DIR, 'sn_ia_manifest.csv'), index=False)
sn_ib_manifest.to_csv(os.path.join(BASE_DIR, 'sn_ib_manifest.csv'), index=False)
sn_ic_manifest.to_csv(os.path.join(BASE_DIR, 'sn_ic_manifest.csv'), index=False)
sn_ii_manifest.to_csv(os.path.join(BASE_DIR, 'sn_ii_manifest.csv'), index=False)
slsn_manifest.to_csv(os.path.join(BASE_DIR, 'slsn_manifest.csv'), index=False)
agn_manifest.to_csv(os.path.join(BASE_DIR, 'agn_manifest.csv'), index=False)
tde_manifest.to_csv(os.path.join(BASE_DIR, 'tde_manifest.csv'), index=False)

print(f"SN Ia: {len(sn_ia_manifest)}  SN Ib: {len(sn_ib_manifest)}  SN Ic: {len(sn_ic_manifest)}  "
      f"SN II: {len(sn_ii_manifest)}  SLSN: {len(slsn_manifest)}  AGN: {len(agn_manifest)}  TDE: {len(tde_manifest)}")


In [ ]:
FLARE_STAR_NAMES = [
    # --- Standard highly active UV Ceti types ---
    "UV Ceti", "AD Leo", "EV Lac", "YZ CMi", "AU Mic", "Proxima Centauri",
    "GJ 1243", "Ross 154", "Ross 128", "Wolf 359", "Lalande 21185", "Kruger 60",
    "TZ Ari", "GJ 1111", "V1216 Sgr", "EQ Peg", "DO Cep", "V1005 Ori",

    # --- Additional active M-dwarfs and Kepler/TESS targets ---
    "V371 Ori", "WX UMa", "Luyten 726-8", "GJ 896A", "GJ 1156", "GJ 1245A",
    "GJ 3236", "GJ 3338", "GJ 3737", "GJ 3685A", "GJ 424", "GJ 1002",
    "TRAPPIST-1", "LHS 1140", "Teegarden's Star", "GJ 1061", "YZ Cet",
    "Luyten's Star", "Lacaille 8760", "Lacaille 9352", "Gliese 1", "Gliese 876",
    "Gliese 682", "Gliese 832", "Gliese 667 C", "Kepler-411",
]


In [ ]:
# Resumable TESS download — checkpoints after every star.
# target_total stops the whole download once enough flare light curves are
# collected, so this doesn't wildly overshoot the other three classes' sample size.
import lightkurve as lk

def fetch_tess_flares(star_names, out_subdir='stellar_flares', max_sectors_per_star=14, target_total=None):
    out_dir = os.path.join(BASE_DIR, out_subdir)
    manifest_path = os.path.join(BASE_DIR, 'stellar_flares_manifest_partial.csv')

    if os.path.exists(manifest_path):
        manifest = pd.read_csv(manifest_path).to_dict('records')
        done_files = {m['file'] for m in manifest}
        print(f"Resuming flares: {len(done_files)} already done")
    else:
        manifest = []
        done_files = set()

    if target_total is not None and len(manifest) >= target_total:
        print(f"Already at target ({len(manifest)}/{target_total}) — nothing to do.")
        return pd.DataFrame(manifest)

    reached_target = False
    for name in tqdm(star_names, desc='TESS flare stars'):
        if reached_target:
            break
        try:
            search = lk.search_lightcurve(name, mission='TESS', author='SPOC', exptime=120)
        except Exception as e:
            print(f"search failed for {name}: {e}")
            continue
        if len(search) == 0:
            continue
        for i, entry in enumerate(search[:max_sectors_per_star]):
            safe_name = name.replace(' ', '_')
            fname = f"{safe_name}_sector{i}.csv"
            if fname in done_files:
                continue
            try:
                lc = entry.download().remove_nans()
            except Exception:
                continue
            df = lc.to_pandas().reset_index()[['time', 'flux', 'flux_err']]
            df.rename(columns={'time': 'bjd'}, inplace=True)
            df.to_csv(os.path.join(out_dir, fname), index=False)
            manifest.append({'star_name': name, 'file': fname, 'label': 'stellar_flare', 'n_points': len(df)})
            done_files.add(fname)
            del lc, df
            if target_total is not None and len(manifest) >= target_total:
                print(f"Reached target of {target_total} flare light curves — stopping early.")
                reached_target = True
                break
        pd.DataFrame(manifest).to_csv(manifest_path, index=False)
        gc.collect()

    return pd.DataFrame(manifest)

flare_manifest = fetch_tess_flares(FLARE_STAR_NAMES, target_total=N_PER_CLASS)
flare_manifest.to_csv(os.path.join(BASE_DIR, 'stellar_flares_manifest.csv'), index=False)
print(f"Flares: {len(flare_manifest)}")
if len(flare_manifest) < N_PER_CLASS:
    print(f"NOTE: only reached {len(flare_manifest)}/{N_PER_CLASS} — the star list ran out before hitting")
    print("the target. Add more names to FLARE_STAR_NAMES in the cell above and re-run this cell;")
    print("it will resume from where it left off rather than re-downloading.")



In [ ]:
# Sanity-check plots
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

ztf_panels = [
    (sn_ia_manifest, 'sn_ia', 'SN Ia', 'tab:blue'),
    (sn_ib_manifest, 'sn_ib', 'SN Ib', 'tab:green'),
    (sn_ic_manifest, 'sn_ic', 'SN Ic', 'tab:olive'),
    (sn_ii_manifest, 'sn_ii', 'SN II', 'tab:cyan'),
    (slsn_manifest,  'slsn',  'SLSN',  'tab:red'),
    (agn_manifest,   'agn',   'AGN',   'darkorange'),
    (tde_manifest,   'tde',   'TDE',   'purple'),
]
for ax, (manifest, subdir, name, color) in zip(axes, ztf_panels):
    if len(manifest):
        ex = pd.read_csv(os.path.join(BASE_DIR, subdir, manifest.iloc[0]['oid'] + '.csv'))
        ax.errorbar(ex['mjd'], ex['magpsf'], yerr=ex['sigmapsf'], fmt='o', ms=3, color=color)
        ax.invert_yaxis()
        ax.set_title(f"{name}: {manifest.iloc[0]['tns_name']}")

if len(flare_manifest):
    ex = pd.read_csv(os.path.join(BASE_DIR, 'stellar_flares', flare_manifest.iloc[0]['file']))
    axes[7].plot(ex['bjd'], ex['flux'], lw=0.5, color='crimson')
    axes[7].set_title(f"Flare: {flare_manifest.iloc[0]['star_name']}")

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, '..', 'phase2_sample_lightcurves.png'), dpi=150)
plt.show()
plt.close('all')

print("=== Phase 2 summary ===")
print(f"SN Ia: {len(sn_ia_manifest)}  SN Ib: {len(sn_ib_manifest)}  SN Ic: {len(sn_ic_manifest)}  "
      f"SN II: {len(sn_ii_manifest)}  SLSN: {len(slsn_manifest)}  AGN: {len(agn_manifest)}  "
      f"TDE: {len(tde_manifest)}  Flares: {len(flare_manifest)}")


# PHASE 3 — Preprocessing, Feature Extraction & Model Training

GP interpolation downsamples to at most 800 points before fitting — real TESS light
curves run ~13,000-20,000 points/sector, which was the root cause of an earlier OOM
crash (verified directly: 18k points killed instantly, 800 points fits in ~2s).
Both feature-extraction loops checkpoint to disk every 30 objects.


In [ ]:
# Reload Phase 2 outputs (works standalone even after a kernel restart)
BASE_DIR = '/content/drive/MyDrive/BTP_optical_transients/data/raw'
FEATURE_DIR = '/content/drive/MyDrive/BTP_optical_transients/data/features'
os.makedirs(FEATURE_DIR, exist_ok=True)

sn_ia_manifest = pd.read_csv(os.path.join(BASE_DIR, 'sn_ia_manifest.csv'))
sn_ib_manifest = pd.read_csv(os.path.join(BASE_DIR, 'sn_ib_manifest.csv'))
sn_ic_manifest = pd.read_csv(os.path.join(BASE_DIR, 'sn_ic_manifest.csv'))
sn_ii_manifest = pd.read_csv(os.path.join(BASE_DIR, 'sn_ii_manifest.csv'))
slsn_manifest  = pd.read_csv(os.path.join(BASE_DIR, 'slsn_manifest.csv'))
agn_manifest   = pd.read_csv(os.path.join(BASE_DIR, 'agn_manifest.csv'))
tde_manifest   = pd.read_csv(os.path.join(BASE_DIR, 'tde_manifest.csv'))
flare_manifest = pd.read_csv(os.path.join(BASE_DIR, 'stellar_flares_manifest.csv'))

print(f"SN Ia: {len(sn_ia_manifest)}, SN Ib: {len(sn_ib_manifest)}, SN Ic: {len(sn_ic_manifest)}, "
      f"SN II: {len(sn_ii_manifest)}, SLSN: {len(slsn_manifest)}, AGN: {len(agn_manifest)}, "
      f"TDE: {len(tde_manifest)}, Flares: {len(flare_manifest)}")


In [ ]:
# Cleaning helper
def clean_lightcurve(df, value_col, err_col=None, min_points=8, sigma_clip=5.0):
    df = df.dropna(subset=[value_col]).copy()
    if len(df) < min_points:
        return None
    med = df[value_col].median()
    mad = (df[value_col] - med).abs().median() * 1.4826 + 1e-6
    df = df[(df[value_col] - med).abs() < sigma_clip * mad]
    if len(df) < min_points:
        return None
    return df.sort_values(df.columns[0]).reset_index(drop=True)


In [ ]:
# GP interpolation — downsamples before fitting (root-caused fix, see notes above).
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C

MAX_GP_POINTS = 800   # tested: 18k points -> OOM killed instantly; 800 points -> fits in ~2s

def downsample_for_gp(t, y, max_points=MAX_GP_POINTS):
    if len(t) <= max_points:
        return t, y
    idx = np.linspace(0, len(t) - 1, max_points).astype(int)
    return t[idx], y[idx]

def gp_interpolate(t, y, n_points=200):
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)
    t, y = downsample_for_gp(t, y)
    t0 = t.min()
    X = (t - t0).reshape(-1, 1)
    span = max(t.max() - t.min(), 1.0)
    kernel = (C(1.0, (1e-3, 1e3))
              * RBF(length_scale=span / 10, length_scale_bounds=(1e-2, 1e3))
              + WhiteKernel(noise_level=0.05, noise_level_bounds=(1e-5, 2.0)))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=1)
    gp.fit(X, y)
    t_grid_rel = np.linspace(0, t.max() - t0, n_points)
    y_grid, y_std = gp.predict(t_grid_rel.reshape(-1, 1), return_std=True)
    return t_grid_rel + t0, y_grid


In [ ]:
# Shape-feature extractor
def extract_shape_features(t_grid, y_grid):
    peak_val = float(y_grid.max())
    min_val  = float(y_grid.min())
    t_peak   = float(t_grid[np.argmax(y_grid)])
    rise_time  = t_peak - t_grid[0]
    decay_time = t_grid[-1] - t_peak
    amplitude  = peak_val - min_val
    return {'peak_val': peak_val, 'rise_time': rise_time, 'decay_time': decay_time, 'amplitude': amplitude}


In [ ]:
# SN subtypes / AGN / TDE feature extraction (checkpointed, resumable)
def mag_to_relflux(mag, baseline_mag):
    return 10 ** (-0.4 * (mag - baseline_mag))

def process_ztf_object(path, oid, label):
    df = pd.read_csv(path)
    row = {'id': oid, 'label': label, 'survey': 'ZTF'}
    band_curves = {}
    for fid, band_name in [(1, 'g'), (2, 'r')]:
        band_df = clean_lightcurve(df[df['fid'] == fid][['mjd', 'magpsf']], 'magpsf', min_points=6)
        if band_df is None:
            continue
        try:
            t_grid, mag_grid = gp_interpolate(band_df['mjd'].values, band_df['magpsf'].values)
            band_curves[band_name] = (t_grid, mag_grid)
        except Exception:
            continue
    if 'r' in band_curves:
        primary_t, primary_mag = band_curves['r']
    elif 'g' in band_curves:
        primary_t, primary_mag = band_curves['g']
    else:
        return None
    baseline_mag = primary_mag.max()
    relflux = mag_to_relflux(primary_mag, baseline_mag)
    row.update(extract_shape_features(primary_t, relflux))
    if 'g' in band_curves and 'r' in band_curves:
        t_peak_idx = np.argmax(relflux)
        t_peak = primary_t[t_peak_idx]
        g_t, g_mag = band_curves['g']
        r_t, r_mag = band_curves['r']
        row['color_g_r'] = float(np.interp(t_peak, g_t, g_mag) - np.interp(t_peak, r_t, r_mag))
    else:
        row['color_g_r'] = np.nan
    return row

def extract_ztf_features_checkpointed(manifest, label, subdir, checkpoint_every=30):
    ckpt_path = os.path.join(FEATURE_DIR, f'{subdir}_features_partial.csv')
    if os.path.exists(ckpt_path):
        features = pd.read_csv(ckpt_path).to_dict('records')
        done_ids = {f['id'] for f in features}
        print(f"Resuming {label} features: {len(done_ids)} already done")
    else:
        features = []
        done_ids = set()

    for count, (_, r) in enumerate(tqdm(manifest.iterrows(), total=len(manifest), desc=f'{label} features')):
        if r['oid'] in done_ids:
            continue
        path = os.path.join(BASE_DIR, subdir, f"{r['oid']}.csv")
        feat = process_ztf_object(path, r['oid'], label)
        if feat is not None:
            features.append(feat)
        if count % checkpoint_every == 0:
            pd.DataFrame(features).to_csv(ckpt_path, index=False)
            gc.collect()

    pd.DataFrame(features).to_csv(ckpt_path, index=False)
    return features

sn_ia_features = extract_ztf_features_checkpointed(sn_ia_manifest, 'SN_Ia', 'sn_ia')
sn_ib_features = extract_ztf_features_checkpointed(sn_ib_manifest, 'SN_Ib', 'sn_ib')
sn_ic_features = extract_ztf_features_checkpointed(sn_ic_manifest, 'SN_Ic', 'sn_ic')
sn_ii_features = extract_ztf_features_checkpointed(sn_ii_manifest, 'SN_II', 'sn_ii')
slsn_features  = extract_ztf_features_checkpointed(slsn_manifest,  'SLSN',  'slsn')
agn_features   = extract_ztf_features_checkpointed(agn_manifest,   'AGN',   'agn')
tde_features   = extract_ztf_features_checkpointed(tde_manifest,   'TDE',   'tde')

print(f"SN Ia: {len(sn_ia_features)}/{len(sn_ia_manifest)}   SN Ib: {len(sn_ib_features)}/{len(sn_ib_manifest)}   "
      f"SN Ic: {len(sn_ic_features)}/{len(sn_ic_manifest)}   SN II: {len(sn_ii_features)}/{len(sn_ii_manifest)}   "
      f"SLSN: {len(slsn_features)}/{len(slsn_manifest)}   AGN: {len(agn_features)}/{len(agn_manifest)}   "
      f"TDE: {len(tde_features)}/{len(tde_manifest)}")


In [ ]:
# Stellar flare feature extraction (checkpointed, resumable)
def process_tess_object(path, star_name, label):
    df = pd.read_csv(path)
    clean_df = clean_lightcurve(df[['bjd', 'flux']], 'flux', min_points=20)
    if clean_df is None:
        return None
    median_flux = clean_df['flux'].median()
    if median_flux == 0:
        return None
    try:
        t_grid, flux_grid = gp_interpolate(clean_df['bjd'].values, clean_df['flux'].values)
    except Exception:
        return None
    relflux = flux_grid / median_flux
    row = {'id': star_name, 'label': label, 'survey': 'TESS'}
    row.update(extract_shape_features(t_grid, relflux))
    row['color_g_r'] = np.nan
    return row

def extract_flare_features_checkpointed(manifest, checkpoint_every=30):
    ckpt_path = os.path.join(FEATURE_DIR, 'flare_features_partial.csv')
    if os.path.exists(ckpt_path):
        features = pd.read_csv(ckpt_path).to_dict('records')
        done_set = {f['_file'] for f in features}
        print(f"Resuming flare features: {len(done_set)} already done")
    else:
        features = []
        done_set = set()

    for count, (_, r) in enumerate(tqdm(manifest.iterrows(), total=len(manifest), desc='Flare features')):
        if r['file'] in done_set:
            continue
        path = os.path.join(BASE_DIR, 'stellar_flares', r['file'])
        feat = process_tess_object(path, r['star_name'], 'stellar_flare')
        if feat is not None:
            feat['_file'] = r['file']
            features.append(feat)
        if count % checkpoint_every == 0:
            pd.DataFrame(features).to_csv(ckpt_path, index=False)
            gc.collect()

    pd.DataFrame(features).to_csv(ckpt_path, index=False)
    return features

flare_features = extract_flare_features_checkpointed(flare_manifest)
flare_features = [{k: v for k, v in f.items() if k != '_file'} for f in flare_features]
print(f"Flare features: {len(flare_features)} / {len(flare_manifest)}")


In [ ]:
# Combine, impute missing color, save
feature_df = pd.DataFrame(
    sn_ia_features + sn_ib_features + sn_ic_features + sn_ii_features +
    slsn_features + agn_features + tde_features + flare_features
)
feature_df['has_color'] = feature_df['color_g_r'].notna().astype(int)

global_color_median = feature_df.loc[feature_df['has_color'] == 1, 'color_g_r'].median()
class_medians = feature_df.groupby('label')['color_g_r'].transform('median')
feature_df['color_g_r'] = feature_df['color_g_r'].fillna(class_medians).fillna(global_color_median)

feature_df.to_csv(os.path.join(FEATURE_DIR, 'phase3_features.csv'), index=False)
print(feature_df.shape)
print(feature_df['label'].value_counts())
feature_df.head()


In [ ]:
# ============================================================
# TOP-UP LOOP: guarantee exactly N=30 SURVIVING feature rows per
# SN subtype, by pulling from a larger TNS candidate pool and
# trying the NEXT candidate whenever one fails, instead of just
# accepting the loss. Only touches the 5 SN subtypes.
# ============================================================

TARGET_PER_SUBTYPE = 30
CANDIDATE_POOL_SIZE = 200   # SN_Ib/Ic are rarest (347/432 in catalog) — 200 leaves headroom

def build_subtype_to_target(subtype, target=TARGET_PER_SUBTYPE, pool_size=CANDIDATE_POOL_SIZE,
                             sleep=0.15, checkpoint_every=10):
    out_subdir = subtype.lower()
    out_dir = os.path.join(BASE_DIR, out_subdir)
    os.makedirs(out_dir, exist_ok=True)

    manifest_path = os.path.join(BASE_DIR, f'{out_subdir}_manifest_topup.csv')
    features_path = os.path.join(FEATURE_DIR, f'{out_subdir}_features_topup.csv')

    if os.path.exists(manifest_path) and os.path.exists(features_path):
        manifest = pd.read_csv(manifest_path).to_dict('records')
        features = pd.read_csv(features_path).to_dict('records')
        done_names = {m['tns_name'] for m in manifest}
        print(f"Resuming {subtype}: {len(features)}/{target} already survived")
        if len(features) >= target:
            return manifest[:target], features[:target]
    else:
        manifest, features = [], []
        done_names = set()

    pool = tns_full[tns_full['sn_subtype'] == subtype].copy()
    pool = pool.sample(min(pool_size, len(pool)), random_state=42).reset_index(drop=True)
    print(f"{subtype}: candidate pool = {len(pool)} "
          f"(catalog has {len(tns_full[tns_full['sn_subtype']==subtype])} total)")

    pbar = tqdm(total=target, initial=len(features), desc=f'{subtype} (survivors)')
    for count, (_, row) in enumerate(pool.iterrows()):
        if len(features) >= target:
            break
        if row['name'] in done_names:
            continue
        oid = resolve_oid(row)
        if oid is None:
            continue
        try:
            det = alerce.query_detections(oid, format='pandas', sort='mjd')
        except Exception:
            continue
        if det is None or len(det) < 5:
            continue
        det = det[['mjd', 'fid', 'magpsf', 'sigmapsf', 'ra', 'dec']]
        csv_path = os.path.join(out_dir, f"{oid}.csv")
        det.to_csv(csv_path, index=False)

        feat = process_ztf_object(csv_path, oid, subtype)
        if feat is None:
            continue  # GP/cleaning failed -> try the NEXT candidate, don't just eat the loss

        manifest.append({'tns_name': row['name'], 'oid': oid, 'label': subtype,
                          'n_points': len(det), 'redshift': row.get('redshift')})
        features.append(feat)
        done_names.add(row['name'])
        pbar.update(1)
        time.sleep(sleep)

        if count % checkpoint_every == 0:
            pd.DataFrame(manifest).to_csv(manifest_path, index=False)
            pd.DataFrame(features).to_csv(features_path, index=False)
            gc.collect()

    pbar.close()
    pd.DataFrame(manifest).to_csv(manifest_path, index=False)
    pd.DataFrame(features).to_csv(features_path, index=False)

    if len(features) < target:
        print(f"  WARNING: {subtype} only reached {len(features)}/{target} after exhausting "
              f"a pool of {len(pool)} — genuine scarcity, not a bug. Raise CANDIDATE_POOL_SIZE and rerun.")
    else:
        print(f"  {subtype}: reached target {target}/{target}.")
    return manifest, features


sn_ia_manifest_new, sn_ia_features = build_subtype_to_target('SN_Ia')
sn_ib_manifest_new, sn_ib_features = build_subtype_to_target('SN_Ib')
sn_ic_manifest_new, sn_ic_features = build_subtype_to_target('SN_Ic')
sn_ii_manifest_new, sn_ii_features = build_subtype_to_target('SN_II')
slsn_manifest_new,  slsn_features  = build_subtype_to_target('SLSN')

sn_ia_manifest = pd.DataFrame(sn_ia_manifest_new)
sn_ib_manifest = pd.DataFrame(sn_ib_manifest_new)
sn_ic_manifest = pd.DataFrame(sn_ic_manifest_new)
sn_ii_manifest = pd.DataFrame(sn_ii_manifest_new)
slsn_manifest  = pd.DataFrame(slsn_manifest_new)

# Overwrite the original manifests so a later kernel-restart reload (cell 12) picks up the balanced set
sn_ia_manifest.to_csv(os.path.join(BASE_DIR, 'sn_ia_manifest.csv'), index=False)
sn_ib_manifest.to_csv(os.path.join(BASE_DIR, 'sn_ib_manifest.csv'), index=False)
sn_ic_manifest.to_csv(os.path.join(BASE_DIR, 'sn_ic_manifest.csv'), index=False)
sn_ii_manifest.to_csv(os.path.join(BASE_DIR, 'sn_ii_manifest.csv'), index=False)
slsn_manifest.to_csv(os.path.join(BASE_DIR, 'slsn_manifest.csv'), index=False)

print(f"\nFinal survivor counts — SN Ia: {len(sn_ia_features)}  SN Ib: {len(sn_ib_features)}  "
      f"SN Ic: {len(sn_ic_features)}  SN II: {len(sn_ii_features)}  SLSN: {len(slsn_features)}  "
      f"(target {TARGET_PER_SUBTYPE} each, total {5*TARGET_PER_SUBTYPE})")

In [ ]:
# GP sanity check
def plot_gp_sanity_check(oid_or_name, folder, file_col_is_oid=True, value_col='magpsf', is_mag=True):
    fname = f"{oid_or_name}.csv" if file_col_is_oid else oid_or_name
    path = os.path.join(BASE_DIR, folder, fname)
    if not os.path.exists(path):
        print(f"File not found: {path}")
        return
    df = pd.read_csv(path)
    if 'fid' in df.columns:
        # Filter to one band for visualization
        df = df[df['fid'] == 2]
    x_col = 'mjd' if 'mjd' in df.columns else 'bjd'
    clean_df = clean_lightcurve(df[[x_col, value_col]], value_col, min_points=6)
    if clean_df is None:
        print(f"skipped {oid_or_name}: too few points"); return
    t_grid, y_grid = gp_interpolate(clean_df[x_col].values, clean_df[value_col].values)
    plt.figure(figsize=(6, 3))
    plt.scatter(clean_df[x_col], clean_df[value_col], s=15, label='raw data')
    plt.plot(t_grid, y_grid, color='crimson', label='GP fit')
    if is_mag: plt.gca().invert_yaxis()
    plt.title(f"GP sanity check: {oid_or_name}"); plt.legend(); plt.show()
    plt.close('all')

# Check a ZTF object (SN Ia)
if len(sn_ia_manifest):
    plot_gp_sanity_check(sn_ia_manifest.iloc[0]['oid'], 'sn_ia')

# Check a TESS object (Flare)
if len(flare_manifest):
    plot_gp_sanity_check(flare_manifest.iloc[0]['file'], 'stellar_flares',
                          file_col_is_oid=False, value_col='flux', is_mag=False)

In [ ]:
def trim_to_target(df, label_col, target, seed=42):
    trimmed = []
    for label, group in df.groupby(label_col):
        if len(group) < target:
            print(f"WARNING: '{label}' only has {len(group)} surviving objects — short of the "
                  f"{target} target even after oversampling. Increase OVERSAMPLE_FACTOR for this "
                  f"subtype specifically and re-run, or accept {len(group)} for this class.")
            trimmed.append(group)
        else:
            trimmed.append(group.sample(target, random_state=seed))
    return pd.concat(trimmed, ignore_index=True)

# Define the labels list locally to fix the NameError
SN_SUBTYPE_LABELS = ['SN_Ia', 'SN_Ib', 'SN_Ic', 'SN_II', 'SLSN']

sn_mask_trim = feature_df['label'].isin(SN_SUBTYPE_LABELS)
sn_trimmed = trim_to_target(feature_df[sn_mask_trim], 'label', N_PER_SUBTYPE_TARGET)
feature_df = pd.concat([sn_trimmed, feature_df[~sn_mask_trim]], ignore_index=True)

print(feature_df['label'].value_counts())

In [ ]:
# Feature distributions by class
FEATURES = ['peak_val', 'rise_time', 'decay_time', 'amplitude', 'color_g_r']

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, feat in zip(axes, FEATURES):
    for label in feature_df['label'].unique():
        subset = feature_df[feature_df['label'] == label][feat]
        ax.hist(subset, bins=20, alpha=0.5, label=label)
    ax.set_title(feat); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FEATURE_DIR, 'phase3_feature_distributions.png'), dpi=150)
plt.show()
plt.close('all')


## Two-Stage Hierarchical Classification

Instead of one flat 8-class model, this splits the problem into two independent, balanced
classification tasks:

- **Stage 1 (coarse, 4 classes, 150 each):** the 5 SN subtypes are merged back into a single
  `SNe` label, giving a perfectly balanced SNe(150) / AGN(150) / TDE(150) / stellar_flare(150)
  problem.
- **Stage 2 (fine, 5 classes, ~30 each):** operated **only** on the true SN-subtype rows —
  AGN/TDE/flare never enter this stage. This is a **standalone** evaluation (train/test split
  done independently on the SN-only subset, not cascaded from Stage 1's predictions) — it
  answers "given that something really is a supernova, how well can the subtype be told
  apart?" rather than "how well does the full two-stage pipeline do end-to-end including
  Stage 1's mistakes." The latter (cascaded) evaluation is a reasonable follow-up if you want
  it, but is not what's built here.

Both stages use the same three models: Logistic Regression, Random Forest, and Bagging
(bagged decision trees, matching the configuration already benchmarked earlier in this
notebook: `n_estimators=300, max_samples=0.8, max_features=0.7`).


In [ ]:
# Merge the 5 SN subtypes into one 'SNe' label for Stage 1 — Stage 2 keeps the original,
# fine-grained subtype label untouched in the 'label' column.
SN_SUBTYPE_LABELS = ['SN_Ia', 'SN_Ib', 'SN_Ic', 'SN_II', 'SLSN']
feature_df['coarse_label'] = feature_df['label'].apply(lambda l: 'SNe' if l in SN_SUBTYPE_LABELS else l)
print(feature_df['coarse_label'].value_counts())


In [ ]:
# Adaptive cross-validation folds: 5-fold CV needs at least 5 training examples of the
# SMALLEST class. Stage 2's subtypes are only ~30 objects each before an 80/20 split, and TNS
# subtype rarity is real (see Cell 5's own warning above) — if a subtype comes back thinner
# than expected, this reduces folds gracefully instead of hard-crashing cross_val_score.
def safe_cv_folds(y_arr, desired=5):
    counts = pd.Series(y_arr).value_counts()
    min_count = counts.min()
    folds = min(desired, min_count)
    if folds < desired:
        print(f"NOTE: reducing cross-validation folds from {desired} to {folds} because the "
              f"smallest class in this training split only has {min_count} examples.")
    return max(folds, 2)


### Stage 1 — Coarse Classification (SNe / AGN / TDE / Stellar Flare)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

X_s1 = feature_df[FEATURES + ['has_color']].values
y_s1_raw = feature_df['coarse_label'].values
le_s1 = LabelEncoder()
y_s1 = le_s1.fit_transform(y_s1_raw)
print(f"Stage 1 classes ({len(le_s1.classes_)}):", dict(zip(le_s1.classes_, range(len(le_s1.classes_)))))

idx_s1 = np.arange(len(feature_df))
idx_train_s1, idx_test_s1, y_train_s1, y_test_s1 = train_test_split(
    idx_s1, y_s1, test_size=0.2, stratify=y_s1, random_state=42
)
X_train_s1, X_test_s1 = X_s1[idx_train_s1], X_s1[idx_test_s1]

scaler_s1 = StandardScaler()
X_train_s1_scaled = scaler_s1.fit_transform(X_train_s1)
X_test_s1_scaled = scaler_s1.transform(X_test_s1)
print(f"Stage 1 — Train: {X_train_s1.shape}, Test: {X_test_s1.shape}")


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

logreg_s1 = LogisticRegression(max_iter=1000, class_weight='balanced')
cv_lr_s1 = cross_val_score(logreg_s1, X_train_s1_scaled, y_train_s1, cv=safe_cv_folds(y_train_s1), scoring='accuracy')
print(f"[Stage 1] LogReg CV accuracy: {cv_lr_s1.mean():.3f} +/- {cv_lr_s1.std():.3f}")
logreg_s1.fit(X_train_s1_scaled, y_train_s1)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_s1 = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)
cv_rf_s1 = cross_val_score(rf_s1, X_train_s1_scaled, y_train_s1, cv=safe_cv_folds(y_train_s1), scoring='accuracy')
print(f"[Stage 1] Random Forest CV accuracy: {cv_rf_s1.mean():.3f} +/- {cv_rf_s1.std():.3f}")
rf_s1.fit(X_train_s1_scaled, y_train_s1)


In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_s1 = BaggingClassifier(
    estimator=DecisionTreeClassifier(class_weight='balanced', random_state=42),
    n_estimators=300, max_samples=0.8, max_features=0.7, random_state=42, n_jobs=-1
)
cv_bag_s1 = cross_val_score(bag_s1, X_train_s1_scaled, y_train_s1, cv=safe_cv_folds(y_train_s1), scoring='accuracy')
print(f"[Stage 1] Bagging CV accuracy: {cv_bag_s1.mean():.3f} +/- {cv_bag_s1.std():.3f}")
bag_s1.fit(X_train_s1_scaled, y_train_s1)


In [ ]:
from sklearn.svm import SVC

bag_svm = BaggingClassifier(
    # Swapping Decision Trees for SVM
    estimator=SVC(class_weight='balanced', probability=True, random_state=42),
    n_estimators=100, # Lowered from 300 to prevent long training times
    max_samples=0.8,
    max_features=0.7,
    random_state=42,
    n_jobs=-1
)
cv_bag_svm = cross_val_score(bag_svm, X_train_s1_scaled, y_train_s1, cv=safe_cv_folds(y_train_s1), scoring='accuracy')
print(f"[Stage 1] Bagging (SVM) CV accuracy: {cv_bag_svm.mean():.3f} +/- {cv_bag_svm.std():.3f}")
bag_svm.fit(X_train_s1_scaled, y_train_s1)


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

models_s1 = {'Logistic Regression': logreg_s1, 'Random Forest': rf_s1, 'Bagging (Trees)': bag_s1, 'Support Vector Machine': bag_svm}
fig, axes = plt.subplots(1, 4, figsize=(24, 5))
for ax, (name, model) in zip(axes, models_s1.items()):
    y_pred = model.predict(X_test_s1_scaled)
    acc = accuracy_score(y_test_s1, y_pred)
    cm = confusion_matrix(y_test_s1, y_pred, labels=le_s1.transform(le_s1.classes_))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_s1.classes_)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f"{name}\nTest acc: {acc:.3f}")
plt.suptitle("Stage 1 — Coarse Classification (SNe / AGN / TDE / Stellar Flare)")
plt.tight_layout()
plt.savefig(os.path.join(FEATURE_DIR, 'stage1_confusion_matrices.png'), dpi=150)
plt.show()
plt.close('all')

results_s1 = pd.DataFrame({
    'Model': list(models_s1.keys()),
    'CV Accuracy (mean)': [cv_lr_s1.mean(), cv_rf_s1.mean(), cv_bag_s1.mean(), cv_bag_svm.mean()],
    'CV Accuracy (std)':  [cv_lr_s1.std(),  cv_rf_s1.std(),  cv_bag_s1.std(),  cv_bag_svm.std()],
    'Test Accuracy': [accuracy_score(y_test_s1, m.predict(X_test_s1_scaled)) for m in models_s1.values()],
})
results_s1.to_csv(os.path.join(FEATURE_DIR, 'stage1_model_comparison.csv'), index=False)
print(results_s1)

In [ ]:
import joblib
joblib.dump(logreg_s1, os.path.join(FEATURE_DIR, 'stage1_logreg.pkl'))
joblib.dump(rf_s1, os.path.join(FEATURE_DIR, 'stage1_rf.pkl'))
joblib.dump(bag_s1, os.path.join(FEATURE_DIR, 'stage1_bagging.pkl'))
joblib.dump(scaler_s1, os.path.join(FEATURE_DIR, 'stage1_scaler.pkl'))
joblib.dump(le_s1, os.path.join(FEATURE_DIR, 'stage1_label_encoder.pkl'))
np.save(os.path.join(FEATURE_DIR, 'stage1_idx_train.npy'), idx_train_s1)
np.save(os.path.join(FEATURE_DIR, 'stage1_idx_test.npy'), idx_test_s1)
print("Stage 1 complete.")


### Stage 2 — Fine-Grained SN Subtype Classification (standalone evaluation)

Only the true SN-subtype rows (~30 each of SN Ia/Ib/Ic/II, SLSN) enter this stage. Split,
scaling, and model training are all independent of Stage 1.


In [ ]:
sn_only_df = feature_df[feature_df['label'].isin(SN_SUBTYPE_LABELS)].reset_index(drop=True)
print(f"SN-only subset for Stage 2: {len(sn_only_df)} rows")
print(sn_only_df['label'].value_counts())

X_s2 = sn_only_df[FEATURES + ['has_color']].values
y_s2_raw = sn_only_df['label'].values
le_s2 = LabelEncoder()
y_s2 = le_s2.fit_transform(y_s2_raw)
print(f"Stage 2 classes ({len(le_s2.classes_)}):", dict(zip(le_s2.classes_, range(len(le_s2.classes_)))))

idx_s2 = np.arange(len(sn_only_df))
idx_train_s2, idx_test_s2, y_train_s2, y_test_s2 = train_test_split(
    idx_s2, y_s2, test_size=0.2, stratify=y_s2, random_state=42
)
X_train_s2, X_test_s2 = X_s2[idx_train_s2], X_s2[idx_test_s2]

scaler_s2 = StandardScaler()
X_train_s2_scaled = scaler_s2.fit_transform(X_train_s2)
X_test_s2_scaled = scaler_s2.transform(X_test_s2)
print(f"Stage 2 — Train: {X_train_s2.shape}, Test: {X_test_s2.shape}")


In [ ]:
logreg_s2 = LogisticRegression(max_iter=1000, class_weight='balanced')
cv_lr_s2 = cross_val_score(logreg_s2, X_train_s2_scaled, y_train_s2, cv=safe_cv_folds(y_train_s2), scoring='accuracy')
print(f"[Stage 2] LogReg CV accuracy: {cv_lr_s2.mean():.3f} +/- {cv_lr_s2.std():.3f}")
logreg_s2.fit(X_train_s2_scaled, y_train_s2)


In [ ]:
rf_s2 = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)
cv_rf_s2 = cross_val_score(rf_s2, X_train_s2_scaled, y_train_s2, cv=safe_cv_folds(y_train_s2), scoring='accuracy')
print(f"[Stage 2] Random Forest CV accuracy: {cv_rf_s2.mean():.3f} +/- {cv_rf_s2.std():.3f}")
rf_s2.fit(X_train_s2_scaled, y_train_s2)


In [ ]:
bag_s2 = BaggingClassifier(
    estimator=DecisionTreeClassifier(class_weight='balanced', random_state=42),
    n_estimators=300, max_samples=0.8, max_features=0.7, random_state=42, n_jobs=-1
)
cv_bag_s2 = cross_val_score(bag_s2, X_train_s2_scaled, y_train_s2, cv=safe_cv_folds(y_train_s2), scoring='accuracy')
print(f"[Stage 2] Bagging CV accuracy: {cv_bag_s2.mean():.3f} +/- {cv_bag_s2.std():.3f}")
bag_s2.fit(X_train_s2_scaled, y_train_s2)


In [ ]:
models_s2 = {'Logistic Regression': logreg_s2, 'Random Forest': rf_s2, 'Bagging (Trees)': bag_s2}
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (name, model) in zip(axes, models_s2.items()):
    y_pred = model.predict(X_test_s2_scaled)
    acc = accuracy_score(y_test_s2, y_pred)
    cm = confusion_matrix(y_test_s2, y_pred, labels=le_s2.transform(le_s2.classes_))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_s2.classes_)
    disp.plot(ax=ax, cmap='Oranges', colorbar=False, xticks_rotation=45)
    ax.set_title(f"{name}\nTest acc: {acc:.3f}")
plt.suptitle("Stage 2 — Fine SN Subtype Classification (standalone)")
plt.tight_layout()
plt.savefig(os.path.join(FEATURE_DIR, 'stage2_confusion_matrices.png'), dpi=150)
plt.show()
plt.close('all')

results_s2 = pd.DataFrame({
    'Model': list(models_s2.keys()),
    'CV Accuracy (mean)': [cv_lr_s2.mean(), cv_rf_s2.mean(), cv_bag_s2.mean()],
    'CV Accuracy (std)':  [cv_lr_s2.std(),  cv_rf_s2.std(),  cv_bag_s2.std()],
    'Test Accuracy': [accuracy_score(y_test_s2, m.predict(X_test_s2_scaled)) for m in models_s2.values()],
})
results_s2.to_csv(os.path.join(FEATURE_DIR, 'stage2_model_comparison.csv'), index=False)
print(results_s2)


In [ ]:
joblib.dump(logreg_s2, os.path.join(FEATURE_DIR, 'stage2_logreg.pkl'))
joblib.dump(rf_s2, os.path.join(FEATURE_DIR, 'stage2_rf.pkl'))
joblib.dump(bag_s2, os.path.join(FEATURE_DIR, 'stage2_bagging.pkl'))
joblib.dump(scaler_s2, os.path.join(FEATURE_DIR, 'stage2_scaler.pkl'))
joblib.dump(le_s2, os.path.join(FEATURE_DIR, 'stage2_label_encoder.pkl'))
np.save(os.path.join(FEATURE_DIR, 'stage2_idx_train.npy'), idx_train_s2)
np.save(os.path.join(FEATURE_DIR, 'stage2_idx_test.npy'), idx_test_s2)
print("Stage 2 complete.")


# PHASE 4 — Interpretation & Physical Insights (per stage)

### Stage 1 interpretation

In [ ]:
X_cols = FEATURES + ['has_color']

rf_importance_s1 = pd.DataFrame({'feature': X_cols, 'importance': rf_s1.feature_importances_}).sort_values('importance', ascending=False)
plt.figure(figsize=(6, 4))
plt.barh(rf_importance_s1['feature'], rf_importance_s1['importance'], color='seagreen')
plt.gca().invert_yaxis()
plt.xlabel('Gini importance'); plt.title('Stage 1 Random Forest — feature importance')
plt.tight_layout()
plt.savefig(os.path.join(FEATURE_DIR, 'stage1_rf_importance.png'), dpi=150)
plt.show()
plt.close('all')
print(rf_importance_s1)


In [ ]:
coef_matrix_s1 = pd.DataFrame(logreg_s1.coef_, columns=X_cols, index=le_s1.classes_)
mean_abs_coef_s1 = coef_matrix_s1.abs().mean(axis=0).sort_values(ascending=False)
plt.figure(figsize=(6, 4))
plt.barh(mean_abs_coef_s1.index, mean_abs_coef_s1.values, color='steelblue')
plt.gca().invert_yaxis()
plt.xlabel('Mean |standardized coefficient|'); plt.title('Stage 1 Logistic Regression — feature importance')
plt.tight_layout()
plt.savefig(os.path.join(FEATURE_DIR, 'stage1_logreg_importance.png'), dpi=150)
plt.show()
plt.close('all')

comparison_s1 = pd.DataFrame({
    'RF importance': rf_importance_s1.set_index('feature')['importance'],
    'LogReg |coef|': mean_abs_coef_s1
}).sort_values('RF importance', ascending=False)
print(comparison_s1)


In [ ]:
y_pred_rf_s1 = rf_s1.predict(X_test_s1_scaled)
test_df_s1 = feature_df.iloc[idx_test_s1].copy().reset_index(drop=True)
test_df_s1['true_label'] = le_s1.inverse_transform(y_test_s1)
test_df_s1['pred_label'] = le_s1.inverse_transform(y_pred_rf_s1)
test_df_s1['correct'] = test_df_s1['true_label'] == test_df_s1['pred_label']

assert (test_df_s1['true_label'].values == feature_df.iloc[idx_test_s1]['coarse_label'].values).all(), "Stage 1 train/test alignment broken"

misclassified_s1 = test_df_s1[~test_df_s1['correct']]
print(f"[Stage 1] Misclassified: {len(misclassified_s1)} / {len(test_df_s1)} ({100*len(misclassified_s1)/max(len(test_df_s1),1):.1f}%)")
if len(misclassified_s1):
    print(misclassified_s1.groupby(['true_label', 'pred_label']).size().sort_values(ascending=False))
misclassified_s1[['id', 'true_label', 'pred_label'] + FEATURES]


In [ ]:
top2_s1 = comparison_s1.index[:2].tolist()
print("[Stage 1] Visualizing decision boundary on:", top2_s1)

X2_s1 = feature_df[top2_s1].values
y2_s1 = y_s1

viz_rf_s1 = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)
viz_rf_s1.fit(X2_s1, y2_s1)

x_min, x_max = X2_s1[:, 0].min() - 0.5, X2_s1[:, 0].max() + 0.5
y_min, y_max = X2_s1[:, 1].min() - 0.5, X2_s1[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
Z = viz_rf_s1.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7, 6))
plt.contourf(xx, yy, Z, alpha=0.25, cmap='viridis')
for i, label in enumerate(le_s1.classes_):
    mask = y2_s1 == i
    plt.scatter(X2_s1[mask, 0], X2_s1[mask, 1], label=label, s=20, edgecolor='k', linewidth=0.3)
plt.xlabel(top2_s1[0]); plt.ylabel(top2_s1[1])
plt.title('Stage 1 decision boundary (visualization only)')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FEATURE_DIR, 'stage1_decision_boundary.png'), dpi=150)
plt.show()
plt.close('all')


In [ ]:
physics_summary_s1 = feature_df.groupby('coarse_label')[FEATURES].agg(['mean', 'std']).round(3)
physics_summary_s1.to_csv(os.path.join(FEATURE_DIR, 'stage1_physics_summary.csv'))
print(physics_summary_s1)


### Stage 2 interpretation

In [ ]:
rf_importance_s2 = pd.DataFrame({'feature': X_cols, 'importance': rf_s2.feature_importances_}).sort_values('importance', ascending=False)
plt.figure(figsize=(6, 4))
plt.barh(rf_importance_s2['feature'], rf_importance_s2['importance'], color='seagreen')
plt.gca().invert_yaxis()
plt.xlabel('Gini importance'); plt.title('Stage 2 Random Forest — feature importance')
plt.tight_layout()
plt.savefig(os.path.join(FEATURE_DIR, 'stage2_rf_importance.png'), dpi=150)
plt.show()
plt.close('all')
print(rf_importance_s2)


In [ ]:
coef_matrix_s2 = pd.DataFrame(logreg_s2.coef_, columns=X_cols, index=le_s2.classes_)
mean_abs_coef_s2 = coef_matrix_s2.abs().mean(axis=0).sort_values(ascending=False)
plt.figure(figsize=(6, 4))
plt.barh(mean_abs_coef_s2.index, mean_abs_coef_s2.values, color='steelblue')
plt.gca().invert_yaxis()
plt.xlabel('Mean |standardized coefficient|'); plt.title('Stage 2 Logistic Regression — feature importance')
plt.tight_layout()
plt.savefig(os.path.join(FEATURE_DIR, 'stage2_logreg_importance.png'), dpi=150)
plt.show()
plt.close('all')

comparison_s2 = pd.DataFrame({
    'RF importance': rf_importance_s2.set_index('feature')['importance'],
    'LogReg |coef|': mean_abs_coef_s2
}).sort_values('RF importance', ascending=False)
print(comparison_s2)


In [ ]:
y_pred_rf_s2 = rf_s2.predict(X_test_s2_scaled)
test_df_s2 = sn_only_df.iloc[idx_test_s2].copy().reset_index(drop=True)
test_df_s2['true_label'] = le_s2.inverse_transform(y_test_s2)
test_df_s2['pred_label'] = le_s2.inverse_transform(y_pred_rf_s2)
test_df_s2['correct'] = test_df_s2['true_label'] == test_df_s2['pred_label']

assert (test_df_s2['true_label'].values == sn_only_df.iloc[idx_test_s2]['label'].values).all(), "Stage 2 train/test alignment broken"

misclassified_s2 = test_df_s2[~test_df_s2['correct']]
print(f"[Stage 2] Misclassified: {len(misclassified_s2)} / {len(test_df_s2)} ({100*len(misclassified_s2)/max(len(test_df_s2),1):.1f}%)")
if len(misclassified_s2):
    print(misclassified_s2.groupby(['true_label', 'pred_label']).size().sort_values(ascending=False))
misclassified_s2[['id', 'true_label', 'pred_label'] + FEATURES]


In [ ]:
top2_s2 = comparison_s2.index[:2].tolist()
print("[Stage 2] Visualizing decision boundary on:", top2_s2)

X2_s2 = sn_only_df[top2_s2].values
y2_s2 = y_s2

viz_rf_s2 = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)
viz_rf_s2.fit(X2_s2, y2_s2)

x_min, x_max = X2_s2[:, 0].min() - 0.5, X2_s2[:, 0].max() + 0.5
y_min, y_max = X2_s2[:, 1].min() - 0.5, X2_s2[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
Z = viz_rf_s2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7, 6))
plt.contourf(xx, yy, Z, alpha=0.25, cmap='tab10')
for i, label in enumerate(le_s2.classes_):
    mask = y2_s2 == i
    plt.scatter(X2_s2[mask, 0], X2_s2[mask, 1], label=label, s=20, edgecolor='k', linewidth=0.3)
plt.xlabel(top2_s2[0]); plt.ylabel(top2_s2[1])
plt.title('Stage 2 decision boundary (visualization only)')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FEATURE_DIR, 'stage2_decision_boundary.png'), dpi=150)
plt.show()
plt.close('all')


In [ ]:
physics_summary_s2 = sn_only_df.groupby('label')[FEATURES].agg(['mean', 'std']).round(3)
physics_summary_s2.to_csv(os.path.join(FEATURE_DIR, 'stage2_physics_summary.csv'))
print(physics_summary_s2)
print()
print("Two-stage pipeline complete.")


In [ ]:
# The cascade function: Stage 1 decides SNe vs not; only Stage-1-predicted SNe get a
# Stage 2 subtype prediction. Everything else keeps Stage 1's coarse label as the final answer.
def cascade_predict(stage1_model, stage2_model, X_raw, scaler1, scaler2, le1, le2, sn_coarse_label='SNe'):
    X1_scaled = scaler1.transform(X_raw)
    coarse_pred = le1.inverse_transform(stage1_model.predict(X1_scaled))

    final_pred = coarse_pred.copy().astype(object)
    sne_mask = coarse_pred == sn_coarse_label
    if sne_mask.sum() > 0:
        X2_scaled = scaler2.transform(X_raw[sne_mask])
        subtype_pred = le2.inverse_transform(stage2_model.predict(X2_scaled))
        final_pred[sne_mask] = subtype_pred
    return final_pred, coarse_pred, sne_mask

In [ ]:
# Run the cascade on Stage 1's test set — the natural population for this evaluation,
# since it contains a realistic mix of all coarse classes, not just SNe.
# Models are matched by name: LogReg feeds LogReg, RF feeds RF, Bagging feeds Bagging.
cascade_pairs = {
    'Logistic Regression': (logreg_s1, logreg_s2),
    'Random Forest':       (rf_s1, rf_s2),
    'Bagging (Trees)':     (bag_s1, bag_s2),
}

X_test_s1_raw = feature_df.iloc[idx_test_s1][FEATURES + ['has_color']].values
true_label_test_s1 = feature_df.iloc[idx_test_s1]['label'].values          # fine subtype for SNe, coarse label otherwise
true_coarse_test_s1 = feature_df.iloc[idx_test_s1]['coarse_label'].values  # 'SNe' for every SN subtype, else unchanged

cascade_results = {}
for name, (m1, m2) in cascade_pairs.items():
    final_pred, coarse_pred, sne_mask = cascade_predict(m1, m2, X_test_s1_raw, scaler_s1, scaler_s2, le_s1, le_s2)
    cascade_results[name] = {'final_pred': final_pred, 'coarse_pred': coarse_pred, 'sne_mask': sne_mask}
    print(f"{name}: {sne_mask.sum()} / {len(sne_mask)} test objects routed to Stage 2")

In [ ]:
# Headline metrics per pipeline:
# - End-to-end accuracy: final_pred vs the TRUE fine-grained label, over ALL test objects
# - Conditional Stage-2 accuracy: subtype accuracy ONLY among true SNe that Stage 1 correctly routed
# - False-positive leakage: non-SN objects Stage 1 incorrectly sent into Stage 2 (can never be "correct")
# - False-negative loss: true SNe that Stage 1 failed to route to Stage 2 at all
summary_rows = []
for name, r in cascade_results.items():
    final_pred, coarse_pred, sne_mask = r['final_pred'], r['coarse_pred'], r['sne_mask']

    end_to_end_acc = (final_pred == true_label_test_s1).mean()

    true_sne_mask = true_coarse_test_s1 == 'SNe'
    correctly_routed_sne = sne_mask & true_sne_mask
    if correctly_routed_sne.sum() > 0:
        cond_stage2_acc = (final_pred[correctly_routed_sne] == true_label_test_s1[correctly_routed_sne]).mean()
    else:
        cond_stage2_acc = float('nan')

    false_positive_leak = (sne_mask & ~true_sne_mask).sum()
    false_negative_loss = (~sne_mask & true_sne_mask).sum()

    summary_rows.append({
        'Model': name,
        'End-to-end accuracy': round(end_to_end_acc, 3),
        'Stage 2 accuracy (correctly-routed SNe only)': round(cond_stage2_acc, 3),
        'Non-SN objects leaked into Stage 2': int(false_positive_leak),
        'True SNe never reached Stage 2': int(false_negative_loss),
    })

cascade_summary = pd.DataFrame(summary_rows)
cascade_summary.to_csv(os.path.join(FEATURE_DIR, 'cascade_summary.csv'), index=False)
print(cascade_summary.to_string(index=False))

In [ ]:
# Save cascade predictions for reproducibility / later report-writing
for name, r in cascade_results.items():
    safe_name = name.replace(' ', '_').replace('(', '').replace(')', '')
    out_df = feature_df.iloc[idx_test_s1][['id', 'label', 'coarse_label']].copy()
    out_df['final_pred'] = r['final_pred']
    out_df['stage1_pred'] = r['coarse_pred']
    out_df['routed_to_stage2'] = r['sne_mask']
    out_df.to_csv(os.path.join(FEATURE_DIR, f'cascade_predictions_{safe_name}.csv'), index=False)
print("Cascade predictions saved.")

In [ ]:
# Full confusion matrix over ALL original fine-grained classes (5 SN subtypes + AGN/TDE/flare),
# one per cascade pipeline — this is where Stage 1 routing errors and Stage 2 subtype errors
# both show up in a single picture.
all_classes = sorted(feature_df['label'].unique())
n_classes = len(all_classes)
fig_size = max(5, n_classes * 0.9)
fig, axes = plt.subplots(1, 3, figsize=(fig_size * 3, fig_size))

for ax, (name, r) in zip(axes, cascade_results.items()):
    cm = confusion_matrix(true_label_test_s1, r['final_pred'], labels=all_classes)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=all_classes)
    disp.plot(ax=ax, cmap='Purples', colorbar=False, xticks_rotation=60)
    acc = (r['final_pred'] == true_label_test_s1).mean()
    ax.set_title(f"{name}\nEnd-to-end acc: {acc:.3f}")

plt.suptitle("Cascaded (Option B) full-pipeline confusion matrix")
plt.tight_layout()
plt.savefig(os.path.join(FEATURE_DIR, 'cascade_confusion_matrices.png'), dpi=150)
plt.show()
plt.close('all')

In [ ]:
# Option A (standalone/ceiling) vs Option B (cascaded/realistic) — the gap itself is a finding.
comparison_ab = pd.DataFrame({
    'Model': results_s2['Model'],
    'Option A: standalone Stage 2 test accuracy': results_s2['Test Accuracy'].values,
    'Option B: cascaded end-to-end accuracy': cascade_summary['End-to-end accuracy'].values,
    'Option B: Stage 2 accuracy on correctly-routed SNe': cascade_summary['Stage 2 accuracy (correctly-routed SNe only)'].values,
})
comparison_ab['Gap (A − B end-to-end)'] = (comparison_ab['Option A: standalone Stage 2 test accuracy']
                                            - comparison_ab['Option B: cascaded end-to-end accuracy']).round(3)
comparison_ab.to_csv(os.path.join(FEATURE_DIR, 'option_a_vs_b_comparison.csv'), index=False)
print(comparison_ab.to_string(index=False))

In [ ]:
# Fair Option B metric: restricted to the SAME population as Option A (true SNe only),
# counting a Stage-1 miss as wrong. This replaces the "end-to-end over all 95 objects"
# comparison above, which is diluted by the easy non-SN classes and isn't a fair comparison.
fair_b_rows = []
for name, r in cascade_results.items():
    sne_mask, coarse_pred, final_pred = r['sne_mask'], r['coarse_pred'], r['final_pred']
    true_sne_mask = true_coarse_test_s1 == 'SNe'
    correctly_routed = sne_mask & true_sne_mask
    correct_subtype = correctly_routed & (final_pred == true_label_test_s1)
    fair_b_acc = correct_subtype.sum() / true_sne_mask.sum()
    fair_b_rows.append({'Model': name, 'Fair Option B accuracy (true SNe only)': round(fair_b_acc, 3)})

fair_comparison = pd.DataFrame(fair_b_rows).merge(
    comparison_ab[['Model', 'Option A: standalone Stage 2 test accuracy']], on='Model'
)
fair_comparison['Fair gap (A - B)'] = (fair_comparison['Option A: standalone Stage 2 test accuracy']
                                        - fair_comparison['Fair Option B accuracy (true SNe only)']).round(3)
print(fair_comparison.to_string(index=False))

In [ ]:
# Leakage-free Stage 2: retrain excluding any SN object that's in Stage 1's test set,
# so the fair comparison above can't be inflated by Stage 2 having already seen the answer.
sn_ids_in_stage1_test = set(feature_df.iloc[idx_test_s1][feature_df.iloc[idx_test_s1]['coarse_label'] == 'SNe']['id'])
sn_only_leakfree = sn_only_df[~sn_only_df['id'].isin(sn_ids_in_stage1_test)].reset_index(drop=True)
print(f"SN-only pool: {len(sn_only_df)} -> {len(sn_only_leakfree)} after removing Stage-1-test overlap "
      f"({len(sn_only_df) - len(sn_only_leakfree)} objects excluded)")

X_s2_lf = sn_only_leakfree[FEATURES + ['has_color']].values
y_s2_lf_raw = sn_only_leakfree['label'].values
le_s2_lf = LabelEncoder()
y_s2_lf = le_s2_lf.fit_transform(y_s2_lf_raw)

scaler_s2_lf = StandardScaler()
X_s2_lf_scaled = scaler_s2_lf.fit_transform(X_s2_lf)

logreg_s2_lf = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_s2_lf_scaled, y_s2_lf)
rf_s2_lf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42).fit(X_s2_lf_scaled, y_s2_lf)
bag_s2_lf = BaggingClassifier(
    estimator=DecisionTreeClassifier(class_weight='balanced', random_state=42),
    n_estimators=300, max_samples=0.8, max_features=0.7, random_state=42, n_jobs=-1
).fit(X_s2_lf_scaled, y_s2_lf)

cascade_pairs_lf = {
    'Logistic Regression': (logreg_s1, logreg_s2_lf),
    'Random Forest':       (rf_s1, rf_s2_lf),
    'Bagging (Trees)':     (bag_s1, bag_s2_lf),
}
for name, (m1, m2) in cascade_pairs_lf.items():
    final_pred, coarse_pred, sne_mask = cascade_predict(m1, m2, X_test_s1_raw, scaler_s1, scaler_s2_lf, le_s1, le_s2_lf)
    true_sne_mask = true_coarse_test_s1 == 'SNe'
    correctly_routed = sne_mask & true_sne_mask
    correct_subtype = correctly_routed & (final_pred == true_label_test_s1)
    fair_b_lf = correct_subtype.sum() / true_sne_mask.sum()
    print(f"[leakage-free] {name}: fair Option B accuracy = {fair_b_lf:.3f}  (compare to the leaky number above)")